# Docling on Databricks

This notebook provides a walkthrough of using Docling on Databricks, specifically focusing on deploying a custom model serving endpoint that can be used for scaled document parsing within a customer's data plane.

In [0]:
%pip install docling onnxruntime databricks-sdk databricks-ai-bridge pymssql==2.3.8 hf_transfer --upgrade
%restart_python

The configuration is quite simple - we have a list of paths on a volume and an export location.

In [0]:
simple_paths = [
  '/Volumes/shm/default/raw_pdfs/cauli_wingz.pdf',
  
]

ocr_paths = [
  '/Volumes/shm/default/raw_pdfs/ilpa_modified_LPonly2.pdf'
]

# Configure OAuth authentication for volume access
secret_scope_name = "shm"
client_id_key = "sp-id"
client_secret_key = "sp-auth"

## Docling Usage In Notebook

In [0]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PipelineOptions, EasyOcrOptions
from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice

converter = DocumentConverter()
result = converter.convert(ocr_paths[0])
print(result.status)

### Serving Endpoint Deployment

In [0]:
import os
os.environ["DATABRICKS_HOST"] = "https://adb-984752964297111.11.azuredatabricks.net/"
os.environ["DATABRICKS_CLIENT_ID"] = dbutils.secrets.get(scope=secret_scope_name, key=client_id_key)
os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=secret_scope_name, key=client_secret_key)

In [0]:
from docling_parser import DoclingModel
model = DoclingModel()
model.load_context(None)
model.predict(simple_paths)

In [0]:
import mlflow
from mlflow.models.auth_policy import AuthPolicy, SystemAuthPolicy, UserAuthPolicy
from mlflow.models.resources import DatabricksResource

mlflow.set_experiment('/Workspace/Users/scott.mckean@databricks.com/experiments/docling')

with mlflow.start_run() as run:
    logged_model = mlflow.pyfunc.log_model(
        artifact_path="docling_model",
        python_model='./docling_parser.py',
        input_example=simple_paths,
        pip_requirements=[
            'mlflow',
            'docling', 
            'onnxruntime', 
            'pymssql==2.3.8',
            'databricks-ai-bridge',
            'databricks-sdk'
            ],
        registered_model_name="shm.multimodal.docling_model",
    )

In [0]:
%restart_python

In [0]:
import mlflow
import os

secret_scope_name = "shm"
client_id_key = "sp-id"
client_secret_key = "sp-auth"

os.environ["DATABRICKS_HOST"] = "https://adb-984752964297111.11.azuredatabricks.net/"
os.environ["DATABRICKS_CLIENT_ID"] = dbutils.secrets.get(scope=secret_scope_name, key=client_id_key)
os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=secret_scope_name, key=client_secret_key)

mlflow.models.predict(
    model_uri=f"models:/shm.multimodal.docling_model/27",
    input_data=["/Volumes/shm/default/raw_pdfs/cauli_wingz.pdf"],
    env_manager="uv",
)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)
from databricks.sdk.errors import ResourceAlreadyExists

w = WorkspaceClient()
endpoint_name = "docling-document-parser"
version = '27'

env_vars = {
    "DATABRICKS_HOST": "https://adb-984752964297111.11.azuredatabricks.net/",
    "DATABRICKS_CLIENT_ID": "{{secrets/shm/sp-id}}",
    "DATABRICKS_CLIENT_SECRET": "{{secrets/shm/sp-auth}}"
}

try:
    endpoint = w.serving_endpoints.create(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            name=endpoint_name,
            served_entities=[
                ServedEntityInput(
                    name="docling-parser",
                    entity_name="shm.multimodal.docling_model",
                    entity_version=version,
                    workload_size="Small",
                    workload_type=ServingModelWorkloadType.GPU_SMALL,
                    scale_to_zero_enabled=True,
                    environment_vars=env_vars
                )
            ]
        )
    )
except ResourceAlreadyExists:
    endpoint = w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                name="docling-parser",
                entity_name="shm.multimodal.docling_model",
                entity_version=version,
                workload_size="Small",
                workload_type=ServingModelWorkloadType.GPU_SMALL,
                scale_to_zero_enabled=True,
                environment_vars=env_vars
            )
        ]
    )

Test serving endpoint

In [0]:
from mlflow.deployments import get_deploy_client
deploy_client = get_deploy_client('databricks')
response = deploy_client.predict(
    endpoint="docling-parsing-endpoint",
    inputs={["/Volumes/shm/default/raw_pdfs/cauli_wingz.pdf"]}
)